# Inference: HFACS Extract & Classify with fine-tuned Llama-3.1-8B-Instruct (LoRA)

This notebook loads the LoRA adapter you already trained in `lora_finetune_llama3_extract_classify.ipynb` and runs it on a CSV of ASRS narratives, producing the 8 HFACS contributing-factor flags plus `Q1_Error` / `Q2_Violation` / `Final_Class` for each row.

## 1. Before you start

1. **Runtime**: `Runtime > Change runtime type > A100 GPU`.
2. **Adapter already trained**: this notebook loads the adapter saved at `OUTPUT_DIR` (set below) from your previous training run — no retraining happens here.
3. **Upload the CSV to classify** to Drive. For the 200-row RF feature-generation run, upload `data/processed/rf/llm_feature_inference_input_50_per_class.csv` to:
   ```
   MyDrive/llm_hfcas_extract_and_classify/data/llm_feature_inference_input_50_per_class.csv
   ```
   It must contain a narrative column (default: `Report 1_Narrative`; for the RF input file this is `narrative`).
4. **Heads up on runtime**: generation is unbatched, ~5-15 sec/row on an A100. The 200-row RF file takes roughly 15-50 minutes with `LIMIT = None`.</cell id="cell-1">

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("Memory (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)

In [ ]:
!pip install -q -U transformers accelerate peft datasets huggingface_hub

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# --- Config ---
DRIVE_DIR = "/content/drive/MyDrive/llm_hfcas_extract_and_classify"

BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
ADAPTER_DIR = f"{DRIVE_DIR}/outputs/llama3.1-8b-extract-classify-lora"  # adapter from training notebook

INPUT_CSV = f"{DRIVE_DIR}/data/llm_feature_inference_input_50_per_class.csv"
NARRATIVE_COLUMN = "narrative"
RESULTS_CSV = f"{DRIVE_DIR}/outputs/llama3.1-8b-extract-classify-lora/rf_llm_features_50_per_class.csv"

LIMIT = None  # 200 rows total, set to a small number first to test
MAX_NEW_TOKENS = 300

SYSTEM_PROMPT = (
    "You are an HFACS aviation safety analyst. Read the narrative, decide which "
    "contributing-factor categories are present, and classify the unsafe act. "
    "Return only valid JSON in the exact schema given. No commentary outside the JSON."
)

USER_PROMPT_TEMPLATE = """Contributing factor categories (mark Yes if present, No otherwise):
Situational_Factors — external conditions or anomalies: weather, non-weather environment, ATC issues, aircraft equipment problems, or ground/inflight encounters (weather/turbulence, wake vortex, wildlife, FOD, vehicle, object, person, smoke/fire/fumes, passenger electronic device).
Personnel_Factors — crew communication breakdown, troubleshooting failure, or human-machine interface issue.
Condition_of_Operators — confusion, distraction, loss of situational awareness, or other physiological factors.
Inadequate_Supervision — understaffing, deficient manuals, or insufficient training/qualification.
Failed_to_Correct_Problem — aircraft dispatched or operated with a known active MEL item.
Planned_Inappropriate_Operations — time pressure drove a pre-planned departure from safe practice.
Supervisory_Violation — deliberate deviation from company policy by crew or management.
Organizational_Process — deficient procedures, software/automation, equipment/tooling, ATC/nav facilities, charts/publications, logbook entries, parts, or aircraft/airport/airspace structure issues.

Classify the unsafe act:
Q1_Error — concrete unintentional unsafe act: airborne/ground conflict, NMAC, altitude/speed/track deviation, loss of aircraft control, VFR into IMC, CFIT/CFTT, fuel issue, or gear-up landing?
Q2_Violation — procedural, regulatory, policy, clearance, FAR, airspace, MEL/CDL, weight-and-balance, hazardous-material, passenger-misconduct, or published-material noncompliance (intent not required)?
Final_Class derivation: Q1=Yes & Q2=No -> Error | Q1=No & Q2=Yes -> Violation | both No -> Neither | both Yes -> Both

Return only valid JSON:
{"Situational_Factors":"Yes/No","Personnel_Factors":"Yes/No","Condition_of_Operators":"Yes/No","Inadequate_Supervision":"Yes/No","Failed_to_Correct_Problem":"Yes/No","Planned_Inappropriate_Operations":"Yes/No","Supervisory_Violation":"Yes/No","Organizational_Process":"Yes/No","Q1_Error":"Yes/No","Q2_Violation":"Yes/No","Final_Class":"Error/Violation/Neither/Both"}

Narrative:
{narrative}"""</cell id="cell-5">

In [ ]:
from huggingface_hub import login
login()  # paste your HF read token when prompted

## 2. Load fine-tuned model (base + LoRA adapter)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16, device_map="auto")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

## 3. Load narratives and build prompts

In [ ]:
import pandas as pd

df = pd.read_csv(INPUT_CSV, low_memory=False)
if LIMIT is not None:
    df = df.head(LIMIT).copy()

print(f"Rows to classify: {len(df)}")
print(df[NARRATIVE_COLUMN].iloc[0][:500])

## 4. Run inference

In [ ]:
import json
from tqdm import tqdm

def generate(narrative):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT_TEMPLATE.replace("{narrative}", str(narrative))},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

OUTPUT_FIELDS = [
    "Situational_Factors", "Personnel_Factors", "Condition_of_Operators",
    "Inadequate_Supervision", "Failed_to_Correct_Problem", "Planned_Inappropriate_Operations",
    "Supervisory_Violation", "Organizational_Process", "Q1_Error", "Q2_Violation", "Final_Class",
]

records = []
for narrative in tqdm(df[NARRATIVE_COLUMN]):
    gen_text = generate(narrative)
    try:
        pred = json.loads(gen_text)
    except json.JSONDecodeError:
        pred = {}
    record = {field: pred.get(field, "INVALID") for field in OUTPUT_FIELDS}
    record["raw_output"] = gen_text
    records.append(record)

## 5. Save results

In [ ]:
results_df = pd.concat([df.reset_index(drop=True), pd.DataFrame(records)], axis=1)
results_df.to_csv(RESULTS_CSV, index=False)
print(f"Saved {len(results_df)} classified rows to: {RESULTS_CSV}")
results_df[["Final_Class", "Q1_Error", "Q2_Violation"]].head()